# Spark Structured Streaming from Kafka

## Objective

This notebook connects Apache Spark Structured Streaming to Kafka and consumes real-time financial transactions.

The incoming transaction stream will later be used for real-time fraud detection.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

## Spark Session Setup

Initialize Apache Spark and configure the environment for Structured Streaming.

In [2]:
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark = (
    SparkSession.builder
    .appName("FraudStreaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Started")

Spark Started


## Connect to Kafka

Connect Spark Structured Streaming to the Kafka broker and subscribe to the transaction topic.

In [5]:
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "latest")
    .load()
)

print("Connected to Kafka Successfully")

AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

In [ ]:
transactions = kafka_df.selectExpr(
    "CAST(value AS STRING) as transaction"
)

In [ ]:
query = (
    transactions.writeStream
    .outputMode("append")
    .format("console")
    .start()
)

query.awaitTermination()